# 🔬 Notebook 3: Google Calendar — Deep Dive

## 🛠️ Setup

```bash
cd 06-system-designs/google-calendar
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Deep dive 1

### Recurrence expansion

We never materialize infinite recurring occurrences. Instead, given a query window `[from, to]`, we expand the RRULE lazily. Simple toy expander for `FREQ=DAILY;INTERVAL=n`:

In [ ]:
from datetime import datetime, timedelta, timezone

def expand_daily(start, interval_days, window_from, window_to):
    out = []
    t = start
    while t < window_from:
        t += timedelta(days=interval_days)
    while t <= window_to:
        out.append(t)
        t += timedelta(days=interval_days)
    return out

start = datetime(2026, 1, 1, 9, tzinfo=timezone.utc)
occ = expand_daily(start, 2,
                   datetime(2026,1,5,tzinfo=timezone.utc),
                   datetime(2026,1,15,tzinfo=timezone.utc))
for x in occ: print(x)

## Deep dive 2

### Overriding one occurrence

Users often change *just one* instance of a recurring event ("move next Monday's 1:1 to Tuesday"). We model this with a base event + a small override table keyed by `(event_id, original_start)`.

In [ ]:
from datetime import datetime, timedelta, timezone

base = {"id": 1, "start": datetime(2026,5,4,15,tzinfo=timezone.utc), "rrule_days": 7}
overrides = {}   # original_start → override dict or None (cancelled)

def move(original_start, new_start):
    overrides[original_start] = {"start": new_start}

def cancel(original_start):
    overrides[original_start] = None

def occurrences(window_from, window_to):
    out, t = [], base["start"]
    while t <= window_to:
        if t >= window_from:
            ov = overrides.get(t, "default")
            if ov is None:
                pass                  # cancelled
            elif ov == "default":
                out.append(t)
            else:
                out.append(ov["start"])
        t += timedelta(days=base["rrule_days"])
    return out

move(datetime(2026,5,11,15,tzinfo=timezone.utc),
     datetime(2026,5,12,15,tzinfo=timezone.utc))
cancel(datetime(2026,5,18,15,tzinfo=timezone.utc))

for o in occurrences(datetime(2026,5,1,tzinfo=timezone.utc),
                      datetime(2026,6,1,tzinfo=timezone.utc)):
    print(o)

## Closing thoughts

- Store times in **UTC**; carry the user's tz for display only.
- Store **RRULEs**, not expanded occurrences. Expand on read.
- Model overrides as a small delta table keyed by `original_start`.